# Web Log Analysis System Demo

This notebook demonstrates the multi-agent RAG system for web log analysis.

## Features
- Multi-agent architecture with LangGraph
- RAG integration with multiple retrieval strategies
- External search capabilities
- Web log analysis and security monitoring


## Dependencies


In [2]:
# Check LangChain version
import langchain
print(f"LangChain version: {langchain.__version__}")

# Check other LangChain packages
try:
    import langchain_core
    print(f"LangChain Core version: {langchain_core.__version__}")
except:
    print("LangChain Core not available")

try:
    import langchain_community
    print(f"LangChain Community version: {langchain_community.__version__}")
except:
    print("LangChain Community not available")

try:
    import langchain_openai
    print(f"LangChain OpenAI version: {langchain_openai.__version__}")
except:
    print("LangChain OpenAI not available")

try:
    import langgraph
    print(f"LangGraph version: {langgraph.__version__}")
except:
    print("LangGraph not available")


LangChain version: 0.3.27
LangChain Core version: 0.3.79
LangChain Community version: 0.3.30
LangChain OpenAI not available
LangGraph not available


In [3]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [4]:
import nest_asyncio
nest_asyncio.apply()

## Simple LangGraph RAG

### Retrieval

#### Data Collection and Processing

In [7]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
import os

# Custom TextLoader that handles Unicode properly
class UnicodeTextLoader(TextLoader):
    def __init__(self, file_path: str, encoding: str = "utf-8"):
        super().__init__(file_path, encoding=encoding)

# Load documents with proper Unicode handling
directory_loader = DirectoryLoader("data/web_incidents", glob="**/*.md", loader_cls=UnicodeTextLoader)

all_knowledge_documents = directory_loader.load()

In [8]:
import langchain
print(langchain.__version__)


0.3.27


In [9]:
import tiktoken
from langchain.text_splitter import RecursiveCharacterTextSplitter

def tiktoken_len(text):
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(
        text,
    )
    return len(tokens)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 750,
    chunk_overlap = 0,
    length_function = tiktoken_len,
)

all_knowledge_chunks = text_splitter.split_documents(all_knowledge_documents)


In [10]:
len(all_knowledge_chunks)

57

#### Embedding Model and Vector Store

In [11]:
from langchain_openai.embeddings import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [12]:
from langchain_community.vectorstores import Qdrant

qdrant_vectorstore = Qdrant.from_documents(
    documents=all_knowledge_chunks,
    embedding=embedding_model,
    location=":memory:"
)

In [13]:
qdrant_retriever = qdrant_vectorstore.as_retriever()

### Augmented

In [14]:
from langchain_core.prompts import ChatPromptTemplate

HUMAN_TEMPLATE = """
You are an expert log analyst. Analyze the following log entries using the provided context about similar incidents.

LOG ENTRIES TO ANALYZE:
{query}

CONTEXT ABOUT SIMILAR INCIDENTS:
{context}

Please provide:
1. Error type and severity
2. Root cause analysis
3. Immediate remediation steps
4. Prevention recommendations

Use the context to provide detailed, actionable analysis.
"""

chat_prompt = ChatPromptTemplate.from_messages([
    ("human", HUMAN_TEMPLATE)
])


### Generation

In [15]:
from langchain_openai import ChatOpenAI

generator_llm = ChatOpenAI(model="gpt-4.1-nano")

### RAG - Retrieval Augmented Generation

In [16]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

class LogAnalysisState(TypedDict):
  log_input: str
  context: List[Document]
  analysis_result: str

def retrieve_log_context(state: LogAnalysisState):
  retrieved_docs = qdrant_retriever.invoke(state["log_input"])
  return {"context" : retrieved_docs}

def analyze_log(state: LogAnalysisState):
  generator_chain = chat_prompt | generator_llm | StrOutputParser()
  analysis_result = generator_chain.invoke({"query" : state["log_input"], "context" : state["context"]})
  return {"analysis_result" : analysis_result}

log_analysis_graph = StateGraph(LogAnalysisState).add_sequence([retrieve_log_context, analyze_log])
log_analysis_graph.add_edge(START, "retrieve_log_context")
compiled_log_analyzer = log_analysis_graph.compile()


In [17]:
# Test the log analysis system with actual log input
sample_log_input = """
[Wed Oct 15 10:30:45.123456 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/sensitive-data.php
[Wed Oct 15 10:30:45.123567 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/admin/dashboard.php
[Wed Oct 15 10:30:45.123678 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/config/database.php
[Wed Oct 15 10:30:45.123789 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/backup/system_backup.sql
"""

result = compiled_log_analyzer.invoke({"log_input": sample_log_input})
print("=== LOG ANALYSIS RESULT ===")
print(result["analysis_result"])


=== LOG ANALYSIS RESULT ===
**Analysis of Log Entries and Incident Context**

---

### 1. Error Type and Severity

- **Type of Errors:**  
  - **Client access denial (403 Forbidden)**: Repeated access attempts to sensitive/admin resources are denied, indicating permission restrictions or potential reconnaissance activity.
  - **Server-side failures (500 Internal Server Error and Fatal Errors)**: Indicate application crashes, misconfigurations, or resource exhaustion affecting service availability.
  - **Forbidden errors (403) on admin APIs**: Suggest improper or malicious access attempts to privileged endpoints.
  - **Resource related issues (Memory errors, disk full, permission denied)**: Severe, impacting server stability and availability.

- **Severity Level:**  
  - **High**: Failures leading to server crashes, resource exhaustion, or potential denial of service. The repeated 500 errors, fatal errors, and permission denials are critical issues.
  - **Medium**: Unauthorized access a

In [15]:
# Test Scenario 2: Apache 504 Gateway Timeout
sample_log_504 = """
[Wed Oct 15 14:22:15.456789 2023] [error] [pid 9876:tid 140123456789012] [client 10.0.0.50:43210] AH01084: Timeout while reading response header from upstream server
[Wed Oct 15 14:22:16.123456 2023] [error] [pid 9876:tid 140123456789013] [client 10.0.0.50:43210] AH01085: Upstream server timed out (110: Connection timed out) while reading response header from upstream
[Wed Oct 15 14:22:17.789012 2023] [error] [pid 9876:tid 140123456789014] [client 10.0.0.50:43210] AH01079: The timeout specified has expired: [client 10.0.0.50:43210] AH01084: Timeout while reading response header from upstream
"""

result_504 = compiled_log_analyzer.invoke({"log_input": sample_log_504})
print("=== 504 GATEWAY TIMEOUT ANALYSIS ===")
print(result_504["analysis_result"])

=== 504 GATEWAY TIMEOUT ANALYSIS ===
**Analysis of Log Entries and Contextual Incident Data**

---

### 1. Error Type and Severity

**Primary Error Type:**  
- **Apache Errors:**  
  - `AH01084: Timeout while reading response header from upstream`  
  - `AH01085: Upstream server timed out (110: Connection timed out)`  
  - `AH01079: The timeout specified has expired`

**Severity:**  
- These are **timeout errors** indicating that Apache, acting as a reverse proxy or gateway, was unable to receive a timely response from the upstream service/server.  
- The occurrence of multiple such errors in rapid succession suggests **critical upstream performance issues** or network latency problems, leading to **service unavailability for clients**.  
- Based on contextual similar incidents (e.g., **504 Gateway Timeout**), this can impact user experience, trigger retries, and may cause cascading failures if unresolved.

---

### 2. Root Cause Analysis

**Key insights derived from logs and context:*

In [18]:
# Test Scenario 3: SSL Certificate Expiry
sample_log_ssl_expiry = """
[Wed Oct 15 16:45:30.111222 2023] [error] [pid 5432:tid 140987654321098] [client 172.16.0.25:56789] AH01976: SSL Library Error: error:14094410:SSL routines:ssl3_read_bytes:sslv3 alert handshake failure
[Wed Oct 15 16:45:30.333444 2023] [error] [pid 5432:tid 140987654321099] [client 172.16.0.25:56789] AH02032: Hostname example.com provided via SNI and hostname www.example.com provided via HTTP are different
[Wed Oct 15 16:45:31.555666 2023] [error] [pid 5432:tid 140987654321100] [client 172.16.0.25:56789] AH01961: SSL Library Error: error:1407742E:SSL routines:SSL23_GET_SERVER_HELLO:tlsv1 alert protocol version
[Wed Oct 15 16:45:32.777888 2023] [warn] [pid 5432:tid 140987654321101] [client 172.16.0.25:56789] AH02032: Certificate verification failed: certificate has expired
"""

result_ssl_expiry = compiled_log_analyzer.invoke({"log_input": sample_log_ssl_expiry})
print("=== SSL CERTIFICATE EXPIRY ANALYSIS ===")
print(result_ssl_expiry["analysis_result"])

=== SSL CERTIFICATE EXPIRY ANALYSIS ===
### Analysis Report of the Provided Log Entries

---

#### 1. Error Type and Severity

| Log Entry Snippet | Error Type | Severity | Justification |
|---------------------|--------------|------------|----------------|
| SSL handshake failure, protocol mismatch, expired certificate, hostname mismatch | SSL/TLS errors, certificate validation failures | **High** | These errors directly affect secure communication, prevent client connections, and compromise security integrity. The presence of multiple SSL-related errors indicates a significant security and uptime risk. |

---

#### 2. Root Cause Analysis

The log entries reveal multiple, related issues consistent with SSL/TLS misconfigurations and certificate lifecycle problems:

- **SSL Handshake Failures & Protocol Mismatch:**  
  Errors like `ssl3_read_bytes:sslv3 alert handshake failure` and `SSL23_GET_SERVER_HELLO:tlsv1 alert protocol version` suggest incompatible SSL/TLS versions or configurati

In [19]:
# Save complete output to a file
result = compiled_log_analyzer.invoke({"log_input": sample_log_input})

with open("complete_analysis.txt", "w") as f:
    f.write("=== COMPLETE LOG ANALYSIS RESULT ===\n")
    f.write(result["analysis_result"])

print("Complete analysis saved to complete_analysis.txt")

Complete analysis saved to complete_analysis.txt


## Task 2: Helper Functions for Agent Graphs

#### Import Wall

In [20]:
from typing import Any, Callable, List, Optional, TypedDict, Union

from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.runnables import Runnable
from langchain_core.tools import BaseTool
from langchain_openai import ChatOpenAI

from langgraph.graph import END, StateGraph

### Agent Node Helper

In [21]:
def log_agent_node(state, agent, name):
    """
    Helper function to wrap log analysis agents into LangGraph nodes.
    """
    result = agent.invoke(state)
    
    # Extract the output from the agent result
    if "output" in result:
        output = result["output"]
    else:
        output = str(result)
    
    # Return messages with the log analysis result
    return {"messages": [HumanMessage(content=output, name=name)]}

### Agent Creation Helper Function

In [22]:
def create_log_analysis_agent(
    llm: ChatOpenAI,
    tools: list,
    system_prompt: str,
) -> str:
    """Create a function-calling agent and add it to the graph."""
    system_prompt += ("\nWork autonomously according to your specialty, using the tools available to you."
    " Do not ask for clarification."
    " Your other team members (and other teams) will collaborate with you with their own specialties."
    " You are chosen for a reason!")
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    agent = create_openai_functions_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools)
    return executor

### Supervisor Helper Function

In [23]:
def create_log_analysis_supervisor(llm: ChatOpenAI, system_prompt, members) -> str:
    """An LLM-based router for log analysis."""
    options = ["FINISH"] + members
    function_def = {
        "name": "route",
        "description": "Select the next role.",
        "parameters": {
            "title": "routeSchema",
            "type": "object",
            "properties": {
                "next": {
                    "title": "Next",
                    "anyOf": [
                        {"enum": options},
                    ],
                },
            },
            "required": ["next"],
        },
    }
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="messages"),
            (
                "system",
                "Given the conversation above, who should act next?"
                " Or should we FINISH? Select one of: {options}",
            ),
        ]
    ).partial(options=str(options), team_members=", ".join(members))
    return (
        prompt
        | llm.bind_functions(functions=[function_def], function_call="route")
        | JsonOutputFunctionsParser()
    )


## Task 3: Log Analysis Team - A LangGraph for Analyzing Web Server Logs

### Tool Creation

In [24]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(max_results=5)

C:\Users\Chandu\AppData\Local\Temp\ipykernel_22732\1911882425.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


Creating a custom tool, however, is very straightforward.

In [25]:
from typing import Annotated, List, Tuple, Union
from langchain_core.tools import tool

@tool
def analyze_web_logs(
    query: Annotated[str, "web server log entries to analyze for incidents and remediation"]
):
    """Use Retrieval Augmented Generation to analyze web server logs and provide incident insights"""
    return compiled_log_analyzer.invoke({"log_input": query})

#### Log Analysis Team State

In [26]:
import functools
import operator

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_openai.chat_models import ChatOpenAI
import functools


class LogAnalysisTeamState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    team_members: List[str]
    next: str


#### Log analysis LLM

In [27]:
log_analysis_llm = ChatOpenAI(model="gpt-4o-mini")

### Log Analysis Agents & Nodes

#### Log Analysis: Search Agent

In [28]:
# Log Analysis Team Agents & Nodes

# Create a specialized log search agent for web log analysis
log_search_agent = create_log_analysis_agent(
    llm=log_analysis_llm,
    tools=[tavily_tool],
    system_prompt="You are a specialized web log search assistant. Your role is to search for up-to-date information, documentation, and solutions related to web server errors, security vulnerabilities, and performance issues using external search engines."
    
)

# Create a node for the log search agent
log_search_node = functools.partial(log_agent_node, agent=log_search_agent, name="LogSearch")

#### Log Analysis: RAG Agent Node

In [29]:
# Create a RAG agent using our existing compiled_log_analyzer
rag_agent = create_log_analysis_agent(
    llm=log_analysis_llm,
    tools=[analyze_web_logs],  # Using our custom RAG tool
    system_prompt="You are a specialized log analysis assistant who can provide detailed information about web server incidents, errors, and remediation steps using our knowledge base."
    
)

# Create a node for the RAG agent
rag_node = functools.partial(log_agent_node, agent=rag_agent, name="LogAnalysisRAG")

#### Log Analysis Supervisor Agent

In [30]:
log_analysis_supervisor_agent = create_log_analysis_supervisor(
    log_analysis_llm,
    """You are a supervisor coordinating log analysis experts.

    Available agents:
    - LogSearch: Searches for up-to-date information about web server errors using external search engines
    - LogAnalysisRAG: Analyzes logs using our internal knowledge base for detailed incident analysis

    Decision logic based on error types:
    
    Use LogAnalysisRAG for:
    - Known Apache error codes: AH01797 (403 Forbidden), AH01084/AH01085 (502 Bad Gateway), 
      AH01078/AH00485 (503 Service Unavailable), AH01079 (504 Gateway Timeout), 
      AH01961/AH02032/AH01976 (SSL errors)
    - Security incidents with sensitive files (admin/, config/, backup/)
    - Pattern analysis and correlation with existing incidents
    - Root cause analysis using our incident knowledge base
    
    Use LogSearch for:
    - Unknown or new error codes not in our knowledge base
    - Multiple different error types requiring external documentation
    - System-wide infrastructure issues needing latest solutions
    - Errors not covered in our internal Apache incident documentation
    
    If analysis is complete, choose FINISH.
    
    Always return a valid routing decision.""",
    ["LogSearch", "LogAnalysisRAG"],
)

C:\Users\Chandu\AppData\Local\Temp\ipykernel_22732\2771139156.py:34: LangChainDeprecationWarning: The method `BaseChatOpenAI.bind_functions` was deprecated in langchain-openai 0.2.1 and will be removed in 1.0.0. Use :meth:`~langchain_openai.chat_models.base.ChatOpenAI.bind_tools` instead.
  | llm.bind_functions(functions=[function_def], function_call="route")


In [31]:
result = log_analysis_supervisor_agent.invoke({
    "messages": [
        HumanMessage(content="The error AH01797 indicates client denied by server configuration.")
    ]
})

print(result)

{'next': 'LogAnalysisRAG'}


### Log Analysis Graph Creation

In [32]:
log_analysis_graph = StateGraph(LogAnalysisTeamState)

log_analysis_graph.add_node("LogSearch", log_search_node)
log_analysis_graph.add_node("LogAnalysisRAG", rag_node)
log_analysis_graph.add_node("LogAnalysisSupervisor", log_analysis_supervisor_agent)

In [33]:
log_analysis_graph.add_edge("LogSearch", "LogAnalysisSupervisor") 
log_analysis_graph.add_edge("LogAnalysisRAG", "LogAnalysisSupervisor")
log_analysis_graph.add_conditional_edges(
    "LogAnalysisSupervisor",
    lambda x: x["next"],
    {"LogSearch": "LogSearch", "LogAnalysisRAG": "LogAnalysisRAG", "FINISH": END},
)
log_analysis_graph.set_entry_point("LogAnalysisSupervisor")

In [34]:
compiled_log_analysis_graph = log_analysis_graph.compile()

In [35]:
print(compiled_log_analysis_graph.get_graph().draw_mermaid())#

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	LogSearch(LogSearch)
	LogAnalysisRAG(LogAnalysisRAG)
	LogAnalysisSupervisor(LogAnalysisSupervisor)
	__end__([<p>__end__</p>]):::last
	LogAnalysisRAG --> LogAnalysisSupervisor;
	LogAnalysisSupervisor -.-> LogAnalysisRAG;
	LogAnalysisSupervisor -.-> LogSearch;
	LogAnalysisSupervisor -. &nbsp;FINISH&nbsp; .-> __end__;
	LogSearch --> LogAnalysisSupervisor;
	__start__ --> LogAnalysisSupervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [36]:
def enter_log_analysis_chain(log_input: str):
    results = {
        "messages": [HumanMessage(content=log_input)]
     
    }
    return results

log_analysis_chain = enter_log_analysis_chain | compiled_log_analysis_graph

In [37]:
# Test with a sample log file
sample_log = """
[Wed Oct 15 10:30:45.123456 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/sensitive-data.php
[Wed Oct 15 10:30:45.123567 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/admin/dashboard.php
[Wed Oct 15 10:30:45.123678 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/config/database.php
[Wed Oct 15 10:30:45.123789 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/backup/system_backup.sql
"""

for s in log_analysis_chain.stream(
    sample_log, {"recursion_limit": 10}
):
    if "__end__" not in s:
        print(s)
        print("---")

{'LogAnalysisSupervisor': {'next': 'LogAnalysisRAG'}}
---
{'LogAnalysisRAG': {'messages': [HumanMessage(content='### Analysis of Provided Log Entries\n\n---\n\n#### 1. Error Type and Severity\n\n**Error Type:**  \nThe logs primarily indicate **access control denial (403 Forbidden)** errors due to server configuration restrictions. Additionally, there\'s a presence of **internal server errors (500 Internal Server Error)**, including fatal PHP errors, application exceptions, database connection failures, and resource exhaustion issues.\n\n**Severity Level:**\n- **High Severity:**  \n  - Unauthorized access attempts to sensitive files (e.g., `sensitive-data.php`, `database.php`) suggest potential reconnaissance or malicious probing.  \n  - Internal server errors that could indicate application misconfiguration or exploitation attempts (e.g., PHP fatal errors, database connection failures, memory leaks).  \n- **Medium to High Severity:**  \n  - Consistent 403 Forbidden errors across multip

In [36]:
# Complex log scenario that should trigger both agents
complex_log = """
[Wed Oct 15 10:30:45.123456 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/admin/config.php
[Wed Oct 15 10:30:46.234567 2023] [error] [pid 12345:tid 140234567890177] [client 192.168.1.100:54321] AH01084: pass request body failed
[Wed Oct 15 10:30:47.345678 2023] [error] [pid 12345:tid 140234567890178] [client 192.168.1.100:54321] AH01085: error reading request body: Connection reset by peer
[Wed Oct 15 10:30:48.456789 2023] [error] [pid 12345:tid 140234567890179] [client 192.168.1.100:54321] AH01079: failed to make connection to backend: 127.0.0.1:8080 (Connection refused)
[Wed Oct 15 10:30:49.567890 2023] [error] [pid 12345:tid 140234567890180] [client 192.168.1.100:54321] AH00898: Error reading from remote server returned by /api/users
[Wed Oct 15 10:30:50.678901 2023] [error] [pid 12345:tid 140234567890181] [client 192.168.1.100:54321] AH01078: server reached MaxRequestWorkers setting, consider raising the MaxRequestWorkers setting
[Wed Oct 15 10:30:51.789012 2023] [error] [pid 12345:tid 140234567890182] [client 192.168.1.100:54321] AH00485: scoreboard is full, not at MaxRequestWorkers
[Wed Oct 15 10:30:52.890123 2023] [error] [pid 12345:tid 140234567890183] [client 192.168.1.100:54321] AH01961: SSL Library Error: error:1407742E:SSL routines:SSL23_GET_SERVER_HELLO:tlsv1 alert protocol version
[Wed Oct 15 10:30:53.901234 2023] [error] [pid 12345:tid 140234567890184] [client 192.168.1.100:54321] AH02032: Hostname 192.168.1.100 provided via SNI and hostname example.com provided via HTTP are different
[Wed Oct 15 10:30:54.012345 2023] [error] [pid 12345:tid 140234567890185] [client 192.168.1.100:54321] AH01976: SSL Library Error: error:14094410:SSL routines:ssl3_read_bytes:sslv3 alert handshake failure
"""

# Test both agents working together
for s in log_analysis_chain.stream(
    complex_log, {"recursion_limit": 15}
):
    if "__end__" not in s:
        print(s)
        print("---")

{'LogAnalysisSupervisor': {'next': 'LogAnalysisRAG'}}
---
{'LogAnalysisRAG': {'messages': [HumanMessage(content="### Detailed Analysis of Log Entries with Contextual Insights\n\n---\n\n#### 1. Error Type and Severity\n\n##### **Errors Identified:**\n\n- **Client Denied by Server Configuration**: *Moderate* — indicates access restrictions.\n- **Pass Request Body Failed, Error Reading Request Body, Connection Reset**: *Moderate* — suggests network or client-side issues.\n- **Failed to Make Connection to Backend (Connection Refused)**: *Critical* — backend service unavailable.\n- **Error Reading from Remote Server**: *Moderate* — communication issue with backend.\n- **Reached MaxRequestWorkers / Scoreboard Full**: *Critical* — server resource exhaustion.\n- **SSL Library Errors (TLS Protocol Version, Handshake Failures)**: *Critical* — SSL misconfigurations or protocol incompatibilities.\n- **Hostname Mismatch, SSL Certificate Errors (Expired, Invalid, Self-Signed, Revoked, Untrusted)**: 

In [37]:
# Try this simpler log to trigger both agents
ambiguous_log = """
[Wed Oct 15 10:30:45.123456 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/sensitive-data.php
[Wed Oct 15 10:30:46.234567 2023] [error] [pid 12345:tid 140234567890177] [client 192.168.1.100:54321] AH01797: client denied by server configuration: /var/www/html/admin/dashboard.php
"""

# Test with simpler log
for s in log_analysis_chain.stream(
    ambiguous_log, {"recursion_limit": 15}
):
    if "__end__" not in s:
        print(s)
        print("---")

{'LogAnalysisSupervisor': {'next': 'LogAnalysisRAG'}}
---
{'LogAnalysisRAG': {'messages': [HumanMessage(content='### Analysis of Log Entries and Incident Context\n\n1. **Error Type and Severity**\n   - **Error Type:** Authorization/Access Denied Errors (HTTP 403 Forbidden)\n   - **Severity:** High — suggests configuration or permission issues that may restrict access to sensitive or administrative resources, potentially indicating misconfigurations or attempted unauthorized access.\n\n2. **Root Cause Analysis**\n   - The log entries indicate attempts by clients (client 192.168.1.100) to access sensitive PHP scripts (`/sensitive-data.php` and `/admin/dashboard.php`) are denied by server configuration.\n   - Similar patterns from the context show multiple instances of "client denied by server configuration" and "access denied" errors when requesting admin or sensitive API endpoints.\n   - **Common Causes Include:**\n     a. Misconfigured directory or file permissions in the server config

In [ ]:
# Unknown/new error codes not in your knowledge base
unknown_error_log = """
[Wed Oct 15 10:30:45.123456 2023] [error] [pid 12345:tid 140234567890176] [client 192.168.1.100:54321] AH99999: unknown error code - service unavailable
[Wed Oct 15 10:30:46.234567 2023] [error] [pid 12345:tid 140234567890177] [client 192.168.1.100:54321] AH88888: new error type - connection timeout
[Wed Oct 15 10:30:47.345678 2023] [error] [pid 12345:tid 140234567890178] [client 192.168.1.100:54321] AH77777: undefined error - memory allocation failed
[Wed Oct 15 10:30:48.456789 2023] [error] [pid 12345:tid 140234567890179] [client 192.168.1.100:54321] AH66666: custom error - database connection pool exhausted
"""

# Test with unknown errors
for s in log_analysis_chain.stream(
    unknown_error_log, {"recursion_limit": 15}
):
    if "__end__" not in s:
        print(s)
        print("---")

{'LogAnalysisSupervisor': {'next': 'LogSearch'}}
---
{'LogSearch': {'messages': [HumanMessage(content='Here are relevant resources for each of the error logs you provided:\n\n1. **Service Unavailable (AH99999)**:\n   - **Resource**: [How to fix the 503 Service Unavailable error](https://www.hostinger.com/tutorials/how-to-fix-503-service-unavailable-error)\n     - This article outlines that a 503 error indicates the server cannot handle the request, often due to maintenance or resource limitations. To troubleshoot, check if your server is undergoing maintenance, monitor resource usage, and ensure sufficient computing power is available.\n   - **Additional Info**: [503 Service Unavailable Errors: Expert Tips & Fixes](https://www.sitelock.com/blog/503-service-unavailable-error-guide/)\n   - **Further Reading**: [503 Service Unavailable Error: What It Is and How to Fix It](https://blog.airbrake.io/blog/http-errors/503-service-unavailable)\n\n2. **Connection Timeout (AH88888)**:\n   - **Res

#### RAGAS Testset Generation Analysis

##### What I Used

RAGAS testset generator with Knowledge Graph approach, using Apache incident markdown files (403, 502, 503, 504, SSL errors) as source documents. Generated 10 test cases with SingleHopSpecific, MultiHopAbstract, and MultiHopSpecific synthesizers.

##### Why It's Not Relevant

Incident documentation contains remediation procedures and response steps, not raw log patterns that SREs actually analyze. Generated tests focus on "what to do when incidents occur" rather than "how to identify patterns in log entries" which is the core log analysis workflow.

##### Impact
Tests generated from incident docs may not reflect actual log analysis scenarios, creating evaluation bias where procedural knowledge is tested instead of pattern recognition skills needed for real log analysis.

In [36]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

c:\Users\Chandu\Documents\test\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Chandu\AppData\Local\Temp\ipykernel_22056\4068800016.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\Chandu\AppData\Local\Temp\ipykernel_22056\4068800016.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-sma

In [39]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [43]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in all_knowledge_documents:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 6, relationships: 0)

In [45]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=all_knowledge_documents, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)

Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 116.80it/s]


In [40]:
#kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 46, relationships: 88)

In [41]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

In [42]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1)
 
]

In [43]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples: 100%|██████████| 10/10 [01:02<00:00,  6.21s/it]


,user_input,reference_contexts,reference,synthesizer_name
0,Apache log errors?,[## 📊 Log Samples ### Web Tier (Apache) Logs `...,The context provides multiple log entries indi...,single_hop_specific_query_synthesizer
1,How does mod_auth help with access control in ...,[# Apache 403 Forbidden — Access Control Viola...,Apache implements access control through vario...,single_hop_specific_query_synthesizer
2,"As an IT System Administrator, what issues rel...",[restore access denied ``` ### Authentication ...,The logs indicate several issues with the Auth...,single_hop_specific_query_synthesizer
3,What is MFA?,[## 🔍 Root Cause Analysis\n\n### Primary Cause...,Multi-factor authentication (MFA) requirements...,single_hop_specific_query_synthesizer
4,What is 192.168.1.0/24 used for?,[## 🔧 Resolution Actions ### Short-term Fixes ...,The context mentions 192.168.1.0/24 as part of...,single_hop_specific_query_synthesizer
5,Apache?,"[user account curl -X POST -H ""Authorization: ...",The provided context does not include informat...,single_hop_specific_query_synthesizer
6,How does Python contribute to a 500 Internal S...,[## 🧩 Overview A 500 Internal Server Error occ...,"In the provided context, Python can cause a 50...",single_hop_specific_query_synthesizer
7,What does the timestamp 2024-01-15T10:00:05.88...,[- Java NullPointerException in ProductService...,The timestamp 2024-01-15T10:00:05.885Z indicat...,single_hop_specific_query_synthesizer
8,How is Java used in managing server infrastruc...,[[EC2:i-0987654321fedcba0] [Service:user-servi...,The context mentions Java in relation to error...,single_hop_specific_query_synthesizer
9,What does innodb_buffer_pool_size do in databa...,[# Apache 500 Internal Server Error — Applicat...,The context indicates that there was a configu...,single_hop_specific_query_synthesizer


In [52]:
# Save testset as CSV
testset_df = testset.to_pandas()
testset_df.to_csv("testset.csv", index=False)

print("✅ Testset saved as testset.csv")

✅ Testset saved as testset.csv


#### LangSmith Integration Analysis

In [45]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [47]:
from langsmith import Client

# Create LangSmith client
client = Client()

print("✅ LangSmith client created successfully!")

✅ LangSmith client created successfully!


In [48]:
for data_row in testset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id="f859d190-688e-4ec3-9596-abb6ebab5b02"
  )

#### Manual Sample Logs 

In [78]:
import pandas as pd

data = [
    {
        "user_input": "[Wed Oct 15 10:30:45 2023] [error] [client 192.168.1.100] AH01797: client denied by server configuration: /var/www/html/admin/config.php",
        "reference": "This log indicates that a client request was blocked by Apache access control. The issue is caused by directory permission or .htaccess restrictions. The resolution is to review Require or AllowOverride directives and update file permissions.",
        "reference_contexts": "403 Forbidden errors occur when Apache denies access due to misconfigured authentication or insufficient permissions. Check access control rules and validate directory ownership.",
        "response": "The user lacks permission to access /admin/config.php. Adjust Apache config or file permissions to allow valid users."
    },
    {
        "user_input": "[Wed Oct 15 11:00:23 2023] [error] [proxy:error] AH01079: failed to make connection to backend: 127.0.0.1:8080 (Connection refused)",
        "reference": "This error means Apache’s proxy module couldn’t connect to the backend application on port 8080. The backend might be down or firewall-restricted. Restart the backend or check the upstream service health.",
        "reference_contexts": "Proxy connection failures typically result from a stopped backend or incorrect port binding. Restart backend service or verify upstream endpoint configuration.",
        "response": "Apache couldn’t reach the backend. Ensure the app on 8080 is running and reachable."
    },
    {
        "user_input": "[Wed Oct 15 11:15:12 2023] [error] [ssl:error] AH02217: ssl_stapling_init_cert: can't retrieve ocsp response for cert",
        "reference": "Apache failed to fetch the OCSP response for the SSL certificate. This happens when the OCSP responder is unreachable or certificate chain is incomplete. Disable OCSP stapling temporarily or fix certificate configuration.",
        "reference_contexts": "SSL stapling errors indicate invalid or misconfigured certificates. Verify that the certificate chain and OCSP responder URLs are valid.",
        "response": "The OCSP response for SSL couldn’t be retrieved. Check OCSP URL or intermediate certificate trust chain."
    },
    {
        "user_input": "[Wed Oct 15 11:35:42 2023] [error] [mpm_event:alert] AH00484: server reached MaxRequestWorkers, consider raising the MaxRequestWorkers setting",
        "reference": "Apache reached its concurrency limit (MaxRequestWorkers). This causes slow responses or request drops. Increase MaxRequestWorkers or add more servers to handle load.",
        "reference_contexts": "When Apache’s worker limit is reached, new requests are queued or rejected. Tune worker limits in mpm_event.conf and monitor concurrency metrics.",
        "response": "Server hit worker limit — scale up workers or optimize concurrency settings."
    },
    {
        "user_input": "[Wed Oct 15 12:00:05 2023] [error] [authz_core:error] AH01630: client denied by server configuration: /var/www/html/private/config.php",
        "reference": "A request attempted to access a protected directory. Apache’s authz_core module denied it. Validate Require all denied or security group rules and ensure no sensitive files are exposed.",
        "reference_contexts": "Authz_core denies access to restricted areas. Ensure sensitive directories (/private, /config) are protected using proper rules.",
        "response": "Access to a restricted file was blocked. No issue unless this request is expected; verify directory access settings."
    }
]

df_manual = pd.DataFrame(data)
df_manual.to_csv("data/manual_log_ragas_testset.csv", index=False)
print("✅ Manual 5-row testset created and saved.")
display(df_manual)


✅ Manual 5-row testset created and saved.


,user_input,reference,reference_contexts,response
0,[Wed Oct 15 10:30:45 2023] [error] [client 192...,This log indicates that a client request was b...,403 Forbidden errors occur when Apache denies ...,The user lacks permission to access /admin/con...
1,[Wed Oct 15 11:00:23 2023] [error] [proxy:erro...,This error means Apache’s proxy module couldn’...,Proxy connection failures typically result fro...,Apache couldn’t reach the backend. Ensure the ...
2,[Wed Oct 15 11:15:12 2023] [error] [ssl:error]...,Apache failed to fetch the OCSP response for t...,SSL stapling errors indicate invalid or miscon...,The OCSP response for SSL couldn’t be retrieve...
3,[Wed Oct 15 11:35:42 2023] [error] [mpm_event:...,Apache reached its concurrency limit (MaxReque...,"When Apache’s worker limit is reached, new req...",Server hit worker limit — scale up workers or ...
4,[Wed Oct 15 12:00:05 2023] [error] [authz_core...,A request attempted to access a protected dire...,Authz_core denies access to restricted areas. ...,Access to a restricted file was blocked. No is...


In [87]:
# ================================================================
# 📘 STEP 1: Import dependencies
# ================================================================
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from ragas.metrics import (
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    LLMContextRecall,
    NoiseSensitivity
)

# ================================================================
# 📗 STEP 2: Ensure correct column formatting
# ================================================================
# Make sure reference_contexts is a list (RAGAS requirement)
df_manual["reference_contexts"] = df_manual["reference_contexts"].apply(
    lambda x: [x] if isinstance(x, str) else x
)

# Simulate retrieval step by copying reference_contexts
# (for real evaluation, replace this with actual retriever output)
df_manual["retrieved_contexts"] = df_manual["reference_contexts"]

# ================================================================
# 📘 STEP 3: Convert to HuggingFace Dataset
# ================================================================
dataset = Dataset.from_pandas(df_manual)

# ================================================================
# 📊 STEP 4: Run Evaluation
# ================================================================
results = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        LLMContextRecall(),
        NoiseSensitivity()]
)

# ================================================================
# 📈 STEP 5: Display Results
# ================================================================
print("✅ RAGAS Evaluation Complete!")
print("📊 Evaluation Results:")
print(results)

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  23%|██▎       | 7/30 [03:09<09:46, 25.49s/it]Exception raised in Job[5]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Evaluating:  70%|███████   | 21/30 [03:37<00:54,  6.06s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[19]: TimeoutError()
Evaluating:  90%|█████████ | 27/30 [05:50<00:40, 13.60s/it]Exception raised in Job[23]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Evaluating: 100%|██████████| 30/30 [06:10<00:00, 12.36

✅ RAGAS Evaluation Complete!
📊 Evaluation Results:
{'faithfulness': 0.5417, 'factual_correctness(mode=f1)': 0.4400, 'answer_relevancy': 0.6646, 'context_entity_recall': 0.2232, 'context_recall': 0.7333, 'noise_sensitivity(mode=relevant)': nan}


#### Synthetic RAG Test Dataset Creation and Metric Evaluation
Running manual test logs through your RAG model to generate a synthetic test dataset (df_rag) , which we can now evaluate using RAGAS metrics or similarity scores to measure the pipeline’s accuracy, relevance, and reliability.

In [117]:
rag_outputs = []

for index, row in df_manual.iterrows():
    log_input = row["user_input"]

    # Run through your RAG pipeline only
    result = compiled_log_analyzer.invoke({"log_input": log_input})

    rag_outputs.append({
        "user_input": log_input,
        "response": result["analysis_result"],
        "retrieved_contexts": [doc.page_content for doc in result["context"]],
        "reference": row["reference"]  # already known from your manual dataset
    })

df_rag = pd.DataFrame(rag_outputs)
display(df_rag)


,user_input,response,retrieved_contexts,reference
0,[Wed Oct 15 10:30:45 2023] [error] [client 192...,**Analysis of Log Entry:**\n\n`[Wed Oct 15 10:...,[2024-01-15T10:00:04.000Z [ERROR] [trace_id:re...,This log indicates that a client request was b...
1,[Wed Oct 15 11:00:23 2023] [error] [proxy:erro...,**Log Analysis and Incident Report**\n\n---\n\...,[```\n2024-01-15T10:00:01.267Z [ERROR] [trace_...,This error means Apache’s proxy module couldn’...
2,[Wed Oct 15 11:15:12 2023] [error] [ssl:error]...,**Analysis of Log Entry:**\n\n`[Wed Oct 15 11:...,[```\n2024-01-15T10:00:01.667Z [ERROR] [trace_...,Apache failed to fetch the OCSP response for t...
3,[Wed Oct 15 11:35:42 2023] [error] [mpm_event:...,**Log Analysis Summary and Recommendations**\n...,[```\n2024-01-15T10:00:01.445Z [WARN] [trace_i...,Apache reached its concurrency limit (MaxReque...
4,[Wed Oct 15 12:00:05 2023] [error] [authz_core...,**Analysis of Log Entry:**\n```\n[Wed Oct 15 1...,[2024-01-15T10:00:04.000Z [ERROR] [trace_id:re...,A request attempted to access a protected dire...


In [118]:
from datasets import Dataset
from ragas.metrics import Faithfulness, ContextEntityRecall, ResponseRelevancy, FactualCorrectness
from ragas import evaluate

df_rag["reference_contexts"] = df_rag["retrieved_contexts"]
dataset = Dataset.from_pandas(df_rag)

results = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall()],
    llm=generator_llm
)

print(results)


Evaluating:  10%|█         | 2/20 [00:57<09:30, 31.69s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  30%|███       | 6/20 [03:27<08:46, 37.59s/it]Exception raised in Job[13]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Evaluating: 100%|██████████| 20/20 [03:42<00:00, 11.11s/it]


{'faithfulness': 0.5811, 'factual_correctness(mode=f1)': 0.9800, 'answer_relevancy': 0.8024, 'context_entity_recall': 0.0333}


#### Advanced Log Retrieval Setup

#### Description:

Initializes log data, embeddings, and language model, then builds multiple advanced retrievers (vector, BM25, multi-query, ensemble, etc.) to support flexible and efficient log search for the RAG pipeline.

In [ ]:
def load_log_data_and_components():
    """Load log incident data and initialize shared components"""
    print("📄 Loading log incident data...")
    
    # Use your existing log documents
    log_docs = all_knowledge_documents  # Your incident documents
    print(f"✅ Using {len(log_docs)} log incident documents")
    
    # Initialize shared components
    log_embeddings = embedding_model  # Your existing embedding model
    chat_model = generator_llm        # Your existing LLM
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)
    
    print("✅ Components initialized")
    return log_docs, log_embeddings, chat_model, child_splitter

def create_all_log_retrievers(log_docs, log_embeddings, chat_model, child_splitter):
    """Create all retrieval strategies for log analytics"""
    print("🔧 Creating all retrieval strategies for log analytics...")
    
    # Create vector store
    log_vectorstore = Qdrant.from_documents(
        documents=log_docs,
        embedding=log_embeddings,
        location=":memory:",
        collection_name="Log_Incident_Knowledge_Base"
    )
    
    # 1. Naive Retriever
    log_naive_retriever = log_vectorstore.as_retriever(search_kwargs={"k": 5})
    
    # 2. BM25 Retriever
    log_bm25_retriever = BM25Retriever.from_documents(log_docs)
    
    # 3. Compression Retriever (optional - skip if no Cohere)
    try:
        compressor = CohereRerank(model="rerank-v3.5")
        log_compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor, 
            base_retriever=log_naive_retriever
        )
    except:
        print("⚠️ Cohere not available, skipping compression retriever")
        log_compression_retriever = log_naive_retriever
    
    # 4. Multi-Query Retriever
    log_multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=log_naive_retriever, 
        llm=chat_model
    )
    
    # 5. Parent Document Retriever
    log_parent_docs = log_docs
    log_parent_client = QdrantClient(location=":memory:")
    log_parent_client.create_collection(
        collection_name="log_full_documents",
        vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
    )
    
    log_parent_document_vectorstore = QdrantVectorStore(
        collection_name="log_full_documents", 
        embedding=log_embeddings, 
        client=log_parent_client
    )
    
    log_parent_store = InMemoryStore()
    log_parent_retriever = ParentDocumentRetriever(
        vectorstore=log_parent_document_vectorstore,
        docstore=log_parent_store,
        child_splitter=child_splitter,
    )
    log_parent_retriever.add_documents(log_parent_docs, ids=None)
    
    # 6. Ensemble Retriever (only include available retrievers)
    available_retrievers = [log_naive_retriever, log_bm25_retriever, log_multi_query_retriever, log_parent_retriever]
    if log_compression_retriever != log_naive_retriever:
        available_retrievers.append(log_compression_retriever)
    
    equal_weighting = [1/len(available_retrievers)] * len(available_retrievers)
    log_ensemble_retriever = EnsembleRetriever(retrievers=available_retrievers, weights=equal_weighting)
    
    # 7. Semantic Chunking
    try:
        semantic_chunker = SemanticChunker(
            log_embeddings,
            breakpoint_threshold_type="percentile"
        )
        log_semantic_documents = semantic_chunker.split_documents(log_docs)
        
        log_semantic_vectorstore = Qdrant.from_documents(
            log_semantic_documents,
            log_embeddings,
            location=":memory:",
            collection_name="Log_Incident_Semantic_Chunks"
        )
        log_semantic_retriever = log_semantic_vectorstore.as_retriever(search_kwargs={"k": 5})
    except:
        print("⚠️ Semantic chunking not available, skipping")
        log_semantic_retriever = log_naive_retriever
    
    print("✅ All log retrievers created")
    
    return {
        'naive': log_naive_retriever,
        'bm25': log_bm25_retriever,
        'compression': log_compression_retriever,
        'multi_query': log_multi_query_retriever,
        'parent': log_parent_retriever,
        'ensemble': log_ensemble_retriever,
        'semantic': log_semantic_retriever,
        'vectorstore': log_vectorstore
    }

In [178]:
# Correct imports for your LangChain version
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain.retrievers import MultiQueryRetriever
from langchain.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain_qdrant import QdrantVectorStore
from langchain.storage import InMemoryStore
from qdrant_client import QdrantClient, models
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
import os
import getpass

os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

# Step 1: Load log data and components
log_docs, log_embeddings, chat_model, child_splitter = load_log_data_and_components()

# Step 2: Create all retrievers
log_retrievers = create_all_log_retrievers(log_docs, log_embeddings, chat_model, child_splitter)

print("Available retrievers:", list(log_retrievers.keys()))

📄 Loading log incident data...
✅ Using 6 log incident documents
✅ Components initialized
🔧 Creating all retrieval strategies for log analytics...
✅ All log retrievers created
Available retrievers: ['naive', 'bm25', 'compression', 'multi_query', 'parent', 'ensemble', 'semantic', 'vectorstore']


In [180]:
# Test with a sample log from your testset
test_log = df_manual['user_input'].iloc[0]
print(f"🔍 Testing with: {test_log}")
print("=" * 80)

# Compare results from different retrievers
print("\n🔍 COMPARISON:")
naive_docs = log_retrievers['naive'].invoke(test_log)
bm25_docs = log_retrievers['bm25'].invoke(test_log)

print(f"Naive retriever: {len(naive_docs)} docs")
print(f"BM25 retriever: {len(bm25_docs)} docs")

# Check content overlap
naive_content = [doc.page_content for doc in naive_docs]
bm25_content = [doc.page_content for doc in bm25_docs]
overlap = len(set(naive_content) & set(bm25_content))
print(f"Content overlap: {overlap} similar documents")

🔍 Testing with: [Wed Oct 15 10:30:45 2023] [error] [client 192.168.1.100] AH01797: client denied by server configuration: /var/www/html/admin/config.php

🔍 COMPARISON:
Naive retriever: 5 docs
BM25 retriever: 4 docs
Content overlap: 3 similar documents


In [185]:
def create_rag_system_for_retriever(retriever, llm):
    """Create a RAG system using a specific retriever"""
    
    def retrieve_log_context(state):
        retrieved_docs = retriever.invoke(state["log_input"])
        return {"context": retrieved_docs}

    def analyze_log(state):
        generator_chain = chat_prompt | llm | StrOutputParser()
        analysis_result = generator_chain.invoke({"query": state["log_input"], "context": state["context"]})
        return {"analysis_result": analysis_result}

    log_analysis_graph = StateGraph(LogAnalysisState).add_sequence([retrieve_log_context, analyze_log])
    log_analysis_graph.add_edge(START, "retrieve_log_context")
    return log_analysis_graph.compile()

print("✅ RAG system creation function added!")

✅ RAG system creation function added!


ERROR! Session/line number was not unique in database. History logging moved to new session 220


In [187]:
# Test creating one RAG system
def create_rag_system_for_retriever(retriever, llm):
    """Create a RAG system using a specific retriever"""
    
    def retrieve_log_context(state):
        retrieved_docs = retriever.invoke(state["log_input"])
        return {"context": retrieved_docs}

    def analyze_log(state):
        generator_chain = chat_prompt | llm | StrOutputParser()
        analysis_result = generator_chain.invoke({"query": state["log_input"], "context": state["context"]})
        return {"analysis_result": analysis_result}

    log_analysis_graph = StateGraph(LogAnalysisState).add_sequence([retrieve_log_context, analyze_log])
    log_analysis_graph.add_edge(START, "retrieve_log_context")
    return log_analysis_graph.compile()

# Test creating one RAG system
print("🔧 Creating RAG system for naive retriever...")
try:
    naive_rag = create_rag_system_for_retriever(log_retrievers['naive'], generator_llm)
    print("✅ Naive RAG system created successfully!")
except Exception as e:
    print(f"Error creating RAG system: {e}")

🔧 Creating RAG system for naive retriever...
✅ Naive RAG system created successfully!


In [188]:
# Test one RAG system
test_log = df_manual['user_input'].iloc[0]
print(f"🔍 Testing RAG system with: {test_log}")

try:
    response = naive_rag.invoke({"log_input": test_log})
    print(f"Response: {response['analysis_result'][:200]}...")
    print(f"Context docs: {len(response['context'])}")
except Exception as e:
    print(f"Error: {e}")

🔍 Testing RAG system with: [Wed Oct 15 10:30:45 2023] [error] [client 192.168.1.100] AH01797: client denied by server configuration: /var/www/html/admin/config.php
Response: 1. **Error Type and Severity:**
   - **Error Type:** Apache client denied by server configuration (HTTP 403 Forbidden)
   - **Severity:** High – indicates an access control restriction preventing auth...
Context docs: 5


#### RAGAS Evaluation for Retrievers

#### Description:

Evaluates multiple log retrievers using RAGAS metrics (faithfulness, context recall, and answer relevancy). Runs each retriever through the RAG pipeline, generates responses, computes scores, and summarizes results to identify the best-performing retriever.

In [214]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall, answer_relevancy
from datasets import Dataset
import numpy as np
import pandas as pd

def evaluate_all_retrievers_simple(log_retrievers, df_manual, llm):
    """
    RAGAS evaluation for multiple retrievers.
    Works with ragas==0.3.7 and uses df_manual with columns:
    ['user_input', 'reference', 'reference_contexts'] (optional)
    """
    results = []
    print("🚀 Starting RAGAS evaluation for all retrievers...")

    # Helper to safely average list outputs
    def mean_if_list(value):
        if isinstance(value, list):
            return float(np.mean(value))
        return float(value)

    for retriever_name, retriever in log_retrievers.items():
        if retriever_name == 'vectorstore':
            continue

        print(f"\n🔍 Evaluating {retriever_name} retriever...")

        try:
            # Step 1: Build the RAG system for this retriever
            rag_system = create_rag_system_for_retriever(retriever, llm)

            # Step 2: Copy test data
            eval_df = df_manual.copy()

            # Step 3: Generate responses using this retriever
            for index, row in eval_df.iterrows():
                log_input = row['user_input']
                response = rag_system.invoke({"log_input": log_input})

                eval_df.at[index, 'response'] = response["analysis_result"]
                eval_df.at[index, 'retrieved_contexts'] = [
                    doc.page_content for doc in response["context"]
                ]

            # Step 4: Prepare dataset for RAGAS
            dataset_list = []
            for _, row in eval_df.iterrows():
                dataset_list.append({
                    "question": row["user_input"],
                    "answer": row["response"],
                    "contexts": row["retrieved_contexts"],
                    "ground_truth": row["reference"],
                    "reference_contexts": row.get("reference_contexts", [])
                })

            ragas_dataset = Dataset.from_list(dataset_list)

            # Step 5: Run RAGAS evaluation
            ragas_result = evaluate(
                ragas_dataset,
                metrics=[faithfulness, context_recall, answer_relevancy]
            )

            # Step 6: Extract and average metrics
            faithfulness_score = mean_if_list(ragas_result["faithfulness"])
            recall_score = mean_if_list(ragas_result["context_recall"])
            relevancy_score = mean_if_list(ragas_result["answer_relevancy"])

            results.append({
                "retriever": retriever_name,
                "faithfulness": faithfulness_score,
                "context_recall": recall_score,
                "answer_relevancy": relevancy_score,
            })

            print(f"✅ {retriever_name} - "
                  f"Faithfulness: {faithfulness_score:.3f}, "
                  f"Recall: {recall_score:.3f}, "
                  f"Relevancy: {relevancy_score:.3f}")

        except Exception as e:
            print(f"❌ Error evaluating {retriever_name}: {e}")

    # Step 7: Print summary table
    if results:
        print("\n📊 RESULTS SUMMARY:")
        print("=" * 65)
        for r in results:
            print(f"{r['retriever']:15} | "
                  f"Faith: {r['faithfulness']:.3f} | "
                  f"Recall: {r['context_recall']:.3f} | "
                  f"Relevancy: {r['answer_relevancy']:.3f}")

        best = max(
            results,
            key=lambda x: (
                x["faithfulness"] + x["context_recall"] + x["answer_relevancy"]
            ) / 3,
        )
        print(f"\n🏆 Best Overall Retriever: {best['retriever']}")
    else:
        print("\n⚠️ No valid results were produced.")

    return results


In [215]:
# 🚀 Run evaluation
results = evaluate_all_retrievers_simple(log_retrievers, df_manual, generator_llm)


🚀 Starting RAGAS evaluation for all retrievers...

🔍 Evaluating naive retriever...


Evaluating: 100%|██████████| 15/15 [02:01<00:00,  8.12s/it]


✅ naive - Faithfulness: 0.802, Recall: 1.000, Relevancy: 0.818

🔍 Evaluating bm25 retriever...


Evaluating: 100%|██████████| 15/15 [02:01<00:00,  8.12s/it]


✅ bm25 - Faithfulness: 0.950, Recall: 1.000, Relevancy: 0.795

🔍 Evaluating compression retriever...


...

✅ parent - Faithfulness: 0.868, Recall: 0.800, Relevancy: 0.806

🔍 Evaluating ensemble retriever...


Evaluating: 100%|██████████| 15/15 [02:06<00:00,  8.45s/it]


✅ ensemble - Faithfulness: 0.973, Recall: 1.000, Relevancy: 0.819

🔍 Evaluating semantic retriever...


Evaluating: 100%|██████████| 15/15 [02:38<00:00, 10.55s/it]


✅ semantic - Faithfulness: 0.902, Recall: 0.867, Relevancy: 0.811

📊 RESULTS SUMMARY:
naive           | Faith: 0.802 | Recall: 1.000 | Relevancy: 0.818
bm25            | Faith: 0.950 | Recall: 1.000 | Relevancy: 0.795
compression     | Faith: 0.922 | Recall: 1.000 | Relevancy: 0.818
multi_query     | Faith: 0.760 | Recall: 1.000 | Relevancy: 0.832
parent          | Faith: 0.868 | Recall: 0.800 | Relevancy: 0.806
ensemble        | Faith: 0.973 | Recall: 1.000 | Relevancy: 0.819
semantic        | Faith: 0.902 | Recall: 0.867 | Relevancy: 0.811

🏆 Best Overall Retriever: ensemble
